# Day 04: Spark Performance Optimization, Partitioning & Broadcast Joins
> **Part 4 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2004%20of%2010-008080?style=for-the-badge)](../README.md)

---

## ⚡ Day 04: Spark Performance Optimization, Partitioning & Broadcast Joins
### 🎯 Learning Objectives:
1. Understand Apache Spark's execution engine: **Lazy Evaluation**, **Transformations (Narrow vs Wide)**, and **Actions**.
2. Master partition management:
   - `repartition(N)`: Full cluster shuffle, redistributing evenly or scaling up partitions.
   - `coalesce(N)`: Shuffle-free partition reduction for efficient writes.
3. Implement **Physical Directory Partitioning** (`partitionBy("order_date")`) and verify **Partition Pruning** using `.explain()`.
4. Avoid the critical anti-pattern: The **Small Files Problem** caused by high-cardinality partition columns.
5. Eliminate heavy network shuffles using **Broadcast Hash Joins** (`broadcast()`) for dimensional lookups.
6. Manage in-memory cache lifecycle with `.cache()`, `.is_cached`, and `.unpersist()`.
7. Conduct empirical query performance benchmarking across partition topologies.
---


### ⚙️ Initialize SparkSession for Day 04


In [114]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Day4_SparkPerformance")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [115]:
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Default parallelism: 8


In [116]:
bronze_df = spark.read.parquet(
    "bronze_incremental"
)

bronze_df.show()

+--------+-----------+--------+--------+-------+----------+--------------------+
|order_id|customer_id| product|quantity|  price|order_date| ingestion_timestamp|
+--------+-----------+--------+--------+-------+----------+--------------------+
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-08 07:30:...|
|    3002|        102|   Mouse|       2|  800.0|2026-09-27|2026-10-08 07:30:...|
|    3003|        103|Keyboard|       1| 1500.0|2026-09-27|2026-10-08 07:30:...|
|    3004|        104| Monitor|       1|12000.0|2026-09-27|2026-10-08 07:30:...|
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-05 08:03:...|
|    3002|        102|   Mouse|       2|  800.0|2026-09-27|2026-10-05 08:03:...|
|    3003|        103|Keyboard|       1| 1500.0|2026-09-27|2026-10-05 08:03:...|
|    3004|        104| Monitor|       1|12000.0|2026-09-27|2026-10-05 08:03:...|
|    3001|        101|  Laptop|       1|55000.0|2026-09-27|2026-10-03 07:56:...|
|    3002|        102|   Mou

In [117]:
print("Rows:", bronze_df.count())

Rows: 156


In [118]:
print(
    "Number of partitions:",
    bronze_df.rdd.getNumPartitions()
)

Number of partitions: 8


In [119]:
large_df = (
    bronze_df
    .withColumn(
        "copy_id",
        explode(
            sequence(
                lit(1),
                lit(1000)
            )
        )
    )
)

In [120]:
large_df.count()

156000

In [121]:
print(
    "Partitions:",
    large_df.rdd.getNumPartitions()
)

Partitions: 8


In [122]:
repartitioned_df = large_df.repartition(8)

In [123]:
print(
    "Partitions:",
    repartitioned_df.rdd.getNumPartitions()
)

Partitions: 8


In [124]:
coalesced_df = large_df.coalesce(2)

In [125]:
print(
    "Partitions:",
    coalesced_df.rdd.getNumPartitions()
)

Partitions: 2


| Operation | Increase partitions | Decrease partitions | Shuffle |
|---|---:|---:|---|
| `repartition()` | ✅ | ✅ | Usually yes |
| `coalesce()` | ❌ | ✅ | Usually avoids full shuffle |


Need MORE partitions?
        ↓
   repartition()

Need FEWER partitions?
        ↓
    coalesce()


In [128]:
repartitioned_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=2662]
         +- *(1) Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@f42e6dce), [order_id#1765, customer_id#1766, product#1767, quantity#1768, price#1769, order_date#1770, ingestion_timestamp#1771], false, [copy_id#1807]
            +- *(1) ColumnarToRow
               +- FileScan parquet [order_id#1765,customer_id#1766,product#1767,quantity#1768,price#1769,order_date#1770,ingestion_timestamp#1771] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Me/Documents/Machine Learning/Pyspark/bronze_incremental], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product:string,quantity:int,price:double,order_date:string,in...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(8

In [129]:
coalesced_df.explain()

== Physical Plan ==
Coalesce 2
+- *(1) Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@f42e6dce), [order_id#1765, customer_id#1766, product#1767, quantity#1768, price#1769, order_date#1770, ingestion_timestamp#1771], false, [copy_id#1807]
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#1765,customer_id#1766,product#1767,quantity#1768,price#1769,order_date#1770,ingestion_timestamp#1771] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Me/Documents/Machine Learning/Pyspark/bronze_incremental], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product:string,quantity:int,price:double,order_date:string,in...




In [130]:
test_df = large_df.filter(
    col("quantity") > 1
)

print("Transformation created")

Transformation created


Has Spark actually processed the data?
No.
Spark uses lazy evaluation.
The actual computation happens when you call an action, such as:


count()
show()
collect()
write


In [133]:
test_df.count()

78000

Transformation
     ↓
Builds execution plan
     ↓
No immediate computation

Action
     ↓
Spark executes plan


Transformations
select()
filter()
withColumn()
groupBy()
join()
repartition()


Actions
count()
show()
collect()
write


In [137]:
filtered_df = large_df.filter(
    col("quantity") > 1
)

In [138]:
filtered_df.count()

78000

In [139]:
filtered_df.groupBy("product").count().show()

+--------+-----+
| product|count|
+--------+-----+
|   Mouse|39000|
|Keyboard|13000|
| Monitor|13000|
|  Laptop|13000|
+--------+-----+



In [140]:
#Cache
filtered_df = large_df.filter(
    col("quantity") > 1
)

filtered_df.cache()

DataFrame[order_id: int, customer_id: int, product: string, quantity: int, price: double, order_date: string, ingestion_timestamp: timestamp, copy_id: int]

In [141]:
filtered_df.count()

78000

In [142]:
print(filtered_df.is_cached)

True


In [143]:
filtered_df.unpersist()

DataFrame[order_id: int, customer_id: int, product: string, quantity: int, price: double, order_date: string, ingestion_timestamp: timestamp, copy_id: int]

In [144]:
print(filtered_df.is_cached)

False


#partitionBy()
PART 13 — partitionBy()
This is different from repartition().
Suppose we want our Parquet data physically organized by:
order_date

We can write:
Cell 15
(    bronze_df    .write    .mode("overwrite")    .partitionBy("order_date")    .parquet("partitioned_orders"))


Now inspect your Jupyter folder.
You'll see something conceptually like:
partitioned_orders/
│
├── order_date=2026-09-27/
│     └── part-....parquet
│
├── order_date=2026-09-28/
│     └── part-....parquet
│
├── order_date=2026-09-29/
│     └── part-....parquet
│
└── order_date=2026-09-30/
      └── part-....parquet

🔥 This is a major Data Engineering concept.
PART 14 — Why partition data?
Imagine you have:
5 TB orders

And someone asks:
Give me orders from September 29.

Without partitioning:
5 TB
 ↓
scan lots of data

With date partitioning:
orders/
   │
   ├── date=2026-09-27
   ├── date=2026-09-28
   ├── date=2026-09-29  ← READ THIS
   └── date=2026-09-30

Spark can potentially avoid reading unrelated partitions.
This is called partition pruning.


In [146]:
(
    bronze_df
    .write
    .mode("overwrite")
    .partitionBy("order_date")
    .parquet("partitioned_orders")
)

In [147]:
#Test partition pruning
partitioned_df = spark.read.parquet(
    "partitioned_orders"
)

In [148]:
sept_29 = partitioned_df.filter(
    col("order_date") == "2026-09-29"
)

sept_29.show()

+--------+-----------+--------+--------+-------+--------------------+----------+
|order_id|customer_id| product|quantity|  price| ingestion_timestamp|order_date|
+--------+-----------+--------+--------+-------+--------------------+----------+
|    3008|        108|Keyboard|       2| 1500.0|2026-10-05 08:38:...|2026-09-29|
|    3009|        109|  Laptop|       1|55000.0|2026-10-05 08:38:...|2026-09-29|
|    3010|        110| Monitor|       2|12000.0|2026-10-05 08:38:...|2026-09-29|
|    3008|        108|Keyboard|       2| 1500.0|2026-10-01 07:21:...|2026-09-29|
|    3009|        109|  Laptop|       1|55000.0|2026-10-01 07:21:...|2026-09-29|
|    3010|        110| Monitor|       2|12000.0|2026-10-01 07:21:...|2026-09-29|
|    3008|        108|Keyboard|       2| 1500.0|2026-10-07 07:57:...|2026-09-29|
|    3009|        109|  Laptop|       1|55000.0|2026-10-07 07:57:...|2026-09-29|
|    3010|        110| Monitor|       2|12000.0|2026-10-07 07:57:...|2026-09-29|
|    3008|        108|Keyboa

In [149]:
sept_29.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [order_id#2116,customer_id#2117,product#2118,quantity#2119,price#2120,ingestion_timestamp#2121,order_date#2122] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Me/Documents/Machine Learning/Pyspark/partitioned_orders], PartitionFilters: [isnotnull(order_date#2122), (order_date#2122 = 2026-09-29)], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product:string,quantity:int,price:double,ingestion_timestamp:...




PART 16 — Important warning: too many partitions
Partitioning isn't automatically good.
Bad:
country
 ↓
200 countries

Potentially okay.
But imagine:
user_id
 ↓
10 million users

You could end up with an enormous number of tiny files.
That's the small files problem.
For example:
10 million users
      ↓
10 million partitions/files

Terrible idea.
So partition columns should generally have sensible cardinality and align with common filtering patterns.
For our project:
order_date

is a reasonable practice column.


In [151]:
#Broadcast join
products = spark.createDataFrame(
    [
        (1, "Laptop", "Electronics"),
        (2, "Mouse", "Accessories"),
        (3, "Keyboard", "Accessories"),
        (4, "Monitor", "Electronics"),
        (5, "Headphones", "Accessories")
    ],
    [
        "product_id",
        "product",
        "category"
    ]
)

In [152]:
large_df

DataFrame[order_id: int, customer_id: int, product: string, quantity: int, price: double, order_date: string, ingestion_timestamp: timestamp, copy_id: int]

In [153]:
products

DataFrame[product_id: bigint, product: string, category: string]

In [154]:
orders_with_products = large_df.join(
    broadcast(products),
    on="product",
    how="left"
)

In [155]:
orders_with_products.show()

+-------+--------+-----------+--------+-------+----------+--------------------+-------+----------+-----------+
|product|order_id|customer_id|quantity|  price|order_date| ingestion_timestamp|copy_id|product_id|   category|
+-------+--------+-----------+--------+-------+----------+--------------------+-------+----------+-----------+
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      1|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      2|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      3|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      4|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      5|         1|Electronics|
| Laptop|    3001|        101|       1|55000.0|2026-09-27|2026-10-08 07:30:...|      6|         1|Electronics|
|

What does broadcast mean?
Instead of:
Large Data
   │
   ├── Worker 1
   ├── Worker 2
   ├── Worker 3
   └── Worker 4

Small Data
   │
   └── shuffle around

Spark can send the small table to workers:
             Small lookup
             /    |    \
            ↓     ↓     ↓
        Worker Worker Worker
            ↑     ↑     ↑
          Large distributed data

This can avoid an expensive shuffle for the small side of a join.
But don't broadcast huge tables blindly.


In [157]:
orders_with_products.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product#1767, order_id#1765, customer_id#1766, quantity#1768, price#1769, order_date#1770, ingestion_timestamp#1771, copy_id#1807, product_id#2146L, category#2148]
   +- BroadcastHashJoin [product#1767], [product#2147], LeftOuter, BuildRight, false, false
      :- Generate explode(org.apache.spark.sql.catalyst.expressions.UnsafeArrayData@f42e6dce), [order_id#1765, customer_id#1766, product#1767, quantity#1768, price#1769, order_date#1770, ingestion_timestamp#1771], false, [copy_id#1807]
      :  +- FileScan parquet [order_id#1765,customer_id#1766,product#1767,quantity#1768,price#1769,order_date#1770,ingestion_timestamp#1771] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Me/Documents/Machine Learning/Pyspark/bronze_incremental], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:int,customer_id:int,product:string,quantity:int,price:double,order_dat

In [158]:
#Create a performance experiment
df_2 = large_df.repartition(2)
df_4 = large_df.repartition(4)
df_8 = large_df.repartition(8)

In [159]:
print("2:", df_2.rdd.getNumPartitions())
print("4:", df_4.rdd.getNumPartitions())
print("8:", df_8.rdd.getNumPartitions())

2: 2
4: 4
8: 8


In [160]:
#Time an operation
import time

In [161]:
start = time.time()

df_2.groupBy("product").count().collect()

end = time.time()

print("Time:", end - start)

Time: 0.7943155765533447


In [162]:
start = time.time()

df_8.groupBy("product").count().collect()

end = time.time()

print("Time:", end - start)

Time: 0.6839978694915771


---
### 🎓 Day 04 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 05: Advanced Transformations, Multi-Table Joins & Data Modeling**](./Day_05_Advanced_Joins_Transformations_and_Data_Modeling.ipynb)
